# Separate data experiments: Pluribus and Leduc

From the late [ACM AI – Poker notebook](https://colab.research.google.com/drive/1GHX6o-6mzOQirpwDJlrc1yZ5bD1CBCCC).
These tasks do not change PokerBench labels or its split. Pluribus records observed
play; Leduc provides an approximate small-game solver distribution. Neither is
interchangeable with full Hold'em GTO recommendation labels.

Both experiments are opt-in. Run solver computation on Colab. They use the optional
`experiments` dependencies; the main training notebook does not need them.

In [ ]:
import runpy
import subprocess
import sys
from pathlib import Path

EXTRAS = "analysis,experiments"

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/poker_decision").is_dir()), None
)
if ROOT is None and Path("/content").is_dir():
    ROOT = Path("/content/Poker-Decision-Model")
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/nmokey/Poker-Decision-Model.git", str(ROOT)],
            check=True,
        )
if ROOT is None:
    raise RuntimeError("Open this notebook inside the project checkout.")
sys.path.insert(0, str(ROOT / "src"))
setup = runpy.run_path(
    str(ROOT / "scripts/notebook_setup.py"), init_globals={"ROOT": ROOT, "EXTRAS": EXTRAS}
)
IN_COLAB = setup["IN_COLAB"]

In [ ]:
import json
import subprocess
import sys
from importlib.metadata import version

import pandas as pd
from IPython.display import display

from poker_decision.source_experiments import (
    download_pluribus,
    generate_leduc,
    prepare_pluribus,
    split_observed_hands,
    write_source_records,
)

RUN_PLURIBUS = False
RUN_LEDUC = False
MAX_HANDS = 10  # Feasibility sample; 10_000 attempts the complete Pluribus collection.
OUT = ROOT / "artifacts/alternative_sources"
OUT.mkdir(parents=True, exist_ok=True)
print("Optional libraries:", {p: version(p) for p in ["pokerkit", "open-spiel"]})

## Pluribus: observed action categories

Download the immutable [PHH v1 archive](https://zenodo.org/records/10796886) only when
enabled; verify its SHA-256, including cached copies. Replay each selected hand
strictly with PokerKit. Build input and legal category menu before reading the next
observed action. Inputs contain only Hero's cards, current public board, public
bets/stacks and the prior public action prefix. Other cards, showdown and future
outcomes are excluded. Exact bet/raise sizing is abstracted away in this experiment.

Save failures and warnings; any replay failure prevents split/export for modeling.
Keep shared hands OR identical contexts together using connected components.
This does not establish session independence or chronological opponent histories.
Preserve publisher attribution and inspect upstream data rights before redistribution;
this notebook does not redistribute the archive.

In [ ]:
if RUN_PLURIBUS:
    archive = download_pluribus(ROOT / "data/raw/alternative_sources")
    audit = prepare_pluribus(archive, max_hands=MAX_HANDS)
    write_source_records(OUT / "pluribus_failures.jsonl", audit["failures"])
    (OUT / "pluribus_audit.json").write_text(
        json.dumps({k: v for k, v in audit.items() if k != "records"}, indent=2) + "\n"
    )
    if audit["failures"]:
        raise RuntimeError("Replay failures saved; resolve them before modeling.")
    bot = [r for r in audit["records"] if r["is_pluribus"]]
    fit, held = split_observed_hands(bot)
    write_source_records(OUT / "pluribus_train.jsonl", fit)
    write_source_records(OUT / "pluribus_validation.jsonl", held)
    display(pd.DataFrame(bot).head())
    print("Observed bot decisions:", len(bot), "| split:", len(fit), len(held))
else:
    print("Pluribus download/replay disabled.")

## Leduc: preserve mixed-policy targets

Two-player Leduc is a simplified game, not Texas Hold'em. Train an approximate CFR+
teacher, record its exact small-game exploitability at checkpoints, and generate
seeded trajectories. Inputs use the acting player's information state. Save the full
teacher action distribution, its argmax, and the sampled action as distinct fields.
Do not call softmax confidence an optimal mixing frequency.

**Exploitability below evaluates only the teacher.** No neural policy is trained or
shown to be unexploitable here. A later neural student should train on distributions
and be evaluated as a complete information-state policy. A data split must explicitly
choose information-state versus episode generalization before student training.

In [ ]:
if RUN_LEDUC:
    if not IN_COLAB:
        raise RuntimeError("Run this solver experiment on Colab.")
    generated = generate_leduc(episodes=2000, iterations=500, seed=42)
    write_source_records(OUT / "leduc_teacher_examples.jsonl", generated["records"])
    (OUT / "leduc_manifest.json").write_text(
        json.dumps({k: v for k, v in generated.items() if k != "records"}, indent=2) + "\n"
    )
    display(pd.DataFrame(generated["convergence"]))
    display(pd.DataFrame(generated["records"]).head())
    print(generated["exploitability_scope"])
else:
    print("Leduc solver computation disabled.")

## Recorded evidence

See the [decision-model results](../docs/week2-results.md) for the completed 40-fit comparison and its original provenance.


## Connect these sources to the decision-model interface

`source_task` adapts audited source rows to the same `DecisionExample` and
`DecisionInputs` consumed by [decision modeling](decision_model.ipynb).
It preserves observed-action versus teacher-argmax semantics, keeps hand/episode
groups outside inference inputs, and provides an explicit question for
`encode_examples(..., question=task.question)`. Retain the source workflow's split
membership. Mixed teacher probabilities remain in the original records; this
adapter does not train on them or pool these labels with PokerBench.


In [ ]:
from poker_decision.tasks import source_task

if RUN_PLURIBUS:
    observed_task = source_task(fit, source="pluribus")
    print(observed_task.question, "|", observed_task.label_semantics)
    print(observed_task.examples[0].inputs)
if RUN_LEDUC:
    teacher_task = source_task(generated["records"], source="leduc_teacher")
    print(teacher_task.question, "|", teacher_task.label_semantics)
    print(teacher_task.examples[0].inputs)